# ViVQA — Main Training Notebook

Notebook này điều phối quá trình huấn luyện và inference cho **hai mô hình**:

| Mô hình | Thư mục | Mô tả |
|---------|---------|-------|
| **Box** | `Box/` | Đa nhiệm: phân loại câu trả lời + dự đoán Bounding Box |
| **V4**  | `V4/`  | Hierarchical VQA với Type Mask theo loại câu hỏi |

**Cấu trúc thư mục:**
```
DHM/
├── main.ipynb          ← Notebook điều phối (file này)
├── utils/
│   ├── preprocessing.py   ← preprocess_vietnamese, find_image_path
│   ├── early_stopping.py  ← EarlyStopping
│   └── fusion.py          ← CrossAttentionFusion
├── Box/
│   ├── config.py          ← CONFIG cho mô hình Box
│   ├── dataset.py         ← ViVQADataset, ViVQAMultiTaskInferenceDataset
│   ├── model.py           ← PhoBERT_BLIP2_EfficientNet_MultiTask
│   ├── trainer.py         ← Vòng lặp huấn luyện
│   └── inference.py       ← Chạy inference và xuất CSV
└── V4/
    ├── config.py          ← CONFIG cho mô hình V4
    ├── dataset.py         ← ViVQADataset, ViVQAInferenceDataset
    ├── model.py           ← PhoBERT_BLIP2_VQA_Hierarchical, create_answer_type_mask
    ├── trainer.py         ← Vòng lặp huấn luyện
    └── inference.py       ← Chạy inference và xuất CSV
```

In [ ]:
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea

In [ ]:
import sys
import os

# Đảm bảo thư mục DHM/ nằm trong Python path để import được các module
notebook_dir = os.path.dirname(os.path.abspath("main.ipynb"))
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

## Mô hình 1 — Box (Đa nhiệm: VQA + Bounding Box)

**Mô tả:** Dùng BLIP-2 (Q-Former) + EfficientNet-B7 + PhoBERT kết hợp qua CrossAttention.  
Hai đầu ra: phân loại câu trả lời và hồi quy tọa độ Bounding Box (chuẩn hóa [0, 1]).  
Có thể chỉnh `CONFIG` trong `Box/config.py`.

In [ ]:
from Box.trainer import train as train_box

train_box()

In [ ]:
from Box.inference import run_inference_and_save as box_inference

box_inference()

## Mô hình 2 — V4 (Hierarchical VQA với Type Mask)

**Mô tả:** Dùng BLIP-2 Vision Encoder (raw features, dim=1408) + PhoBERT kết hợp qua CrossAttention.  
Hai đầu ra: dự đoán loại câu hỏi (type classifier) + phân loại câu trả lời với mask theo từng loại câu hỏi.  
Tập train được tự chia thành Train/Val (90/10) theo stratify `type`.  
Có thể chỉnh `CONFIG` trong `V4/config.py`.

In [ ]:
from V4.trainer import train as train_v4

train_v4()

In [ ]:
from V4.inference import run_inference_and_save as v4_inference

v4_inference()